# Atlas-Reference Model Evaluation

Evaluates the ONNX regression models trained on the **Allen Institute Human Immune Health Atlas 2025**.

> **Concept — Atlas-relative virtual channels:** a cell is flagged **positive** for a functional marker when its *measured* intensity exceeds the model's *prediction* for a healthy reference cell with the same identity-marker profile.

**Sections:**
1. Setup & configuration
2. Model inventory — training metadata per target
3. Model comparison — CV R² vs Test R², win counts, overfit gap
4. Feature map — which identity markers feed each model
5. Held-out atlas evaluation — predicted vs. measured on unseen atlas cells
6. Per-cell-type residual analysis
7. SPT study inference (LUAD progression)
8. Atlas-relative positive calls — distribution comparison
9. Biological sanity check
10. Summary

In [ ]:
from pathlib import Path
import json, os, pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error
from onnxruntime import InferenceSession, SessionOptions
import anndata as ad
from scipy import sparse, stats

sns.set_theme(style='whitegrid', context='notebook', palette='tab10')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT       = Path('..').resolve()
MODELS_DIR      = REPO_ROOT / 'models'
SCRIPTS_DIR     = REPO_ROOT / 'scripts'
ATLAS_PATH      = Path('/Users/gsf/Quantori/MSKCC/smprofiler-data/human_immune_health_atlas_full.h5ad')
ANNOTATIONS     = Path('/Users/gsf/Quantori/MSKCC/smprofiler-data/annotations/channel_annotations.json')
CHANNEL_MAPPING = SCRIPTS_DIR / 'channel_name_mapping.json'
DATASETS_DIR    = Path('/Users/gsf/Quantori/MSKCC/smprofiler-data/datasets')
PRIMARY_STUDY   = 'luad_progression'

# Consistent colour palette for model types (shared across all charts)
MODEL_PALETTE = {
    'random_forest': '#1f77b4',
    'extra_trees':   '#2ca02c',
    'xgboost':       '#d62728',
    'elastic_net':   '#9467bd',
    'ridge':         '#8c564b',
    'huber':         '#e377c2',
    'bayesian_ridge':'#17becf',
}
DEFAULT_COLOR = '#7f7f7f'

print('MODELS_DIR :', MODELS_DIR)
print('Atlas exists:', ATLAS_PATH.exists())
print('Annots exist:', ANNOTATIONS.exists())

## 1. Setup & Configuration

In [ ]:
# ── Channel annotations ──────────────────────────────────────────────────────
def load_channel_annotations(path):
    with open(path) as f:
        data = json.load(f)
    identity   = set(data['groups']['identity']['channels'])
    functional = set()
    for gname, gdata in data['groups'].items():
        if gname not in ('identity', 'tumor'):
            functional.update(gdata['channels'])
    aliases = {a: c for a, c in data.get('aliases', {}).items()
               if isinstance(c, str) and c in identity | functional}
    return identity, functional, aliases

def load_extra_mapping(path):
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        return {k: v for k, v in data.items() if not k.startswith('_')}
    return {}

# ── Model inventory ───────────────────────────────────────────────────────────
def load_model_inventory(models_dir):
    records = []
    for meta_file in sorted(models_dir.glob('**/*.meta.json')):
        with open(meta_file) as f:
            meta = json.load(f)
        base_name = Path(meta_file.stem).stem  # strip both suffixes (.meta.json -> .onnx)
        meta['onnx_path'] = meta_file.parent / (base_name + '.onnx')
        meta['meta_path'] = meta_file
        records.append(meta)
    return records

# ── Atlas channel matching ────────────────────────────────────────────────────
def build_spt_to_atlas(atlas_var_names, all_channels, aliases, extra_mapping):
    spt_upper = {c.upper(): c for c in all_channels}
    atlas_to_spt = {}
    for av in atlas_var_names:
        if av in extra_mapping:
            canonical = extra_mapping[av]
            if canonical in all_channels:
                atlas_to_spt[av] = canonical
                continue
        canonical = aliases.get(av, av)
        if canonical in all_channels:
            atlas_to_spt[av] = canonical
            continue
        if av.upper() in spt_upper:
            atlas_to_spt[av] = spt_upper[av.upper()]
    return {spt: atl for atl, spt in atlas_to_spt.items()}

# ── ONNX inference ────────────────────────────────────────────────────────────
def run_onnx(onnx_path, X):
    opts = SessionOptions()
    opts.log_severity_level = 3
    sess = InferenceSession(str(onnx_path), sess_options=opts)
    return sess.run(None, {'X': X.astype(np.float32)})[0].flatten()

# ── Pickle model loading ──────────────────────────────────────────────────────
def load_pkl(pkl_path):
    with open(pkl_path, 'rb') as f:
        return pickle.load(f)

# ── Per-prediction uncertainty ────────────────────────────────────────────────
def predict_with_std(model, model_name, X_norm):
    """Return (y_mean, y_std).  y_std is None when no per-sample estimate exists."""
    if model_name == 'bayesian_ridge':
        X_scaled = model.named_steps['scaler'].transform(X_norm)
        inner    = model.named_steps['bayesian_ridge']
        return inner.predict(X_scaled, return_std=True)
    if model_name in ('random_forest', 'extra_trees'):
        tree_preds = np.stack([t.predict(X_norm) for t in model.estimators_], axis=0)
        return tree_preds.mean(axis=0), tree_preds.std(axis=0)
    return model.predict(X_norm), None

# ── Colour helper ─────────────────────────────────────────────────────────────
def model_color(model_type):
    return MODEL_PALETTE.get(model_type, DEFAULT_COLOR)

# ── Initialise ────────────────────────────────────────────────────────────────
identity_channels, functional_channels, aliases = load_channel_annotations(ANNOTATIONS)
extra_mapping = load_extra_mapping(CHANNEL_MAPPING)
all_channels  = identity_channels | functional_channels

print(f'Identity channels  : {len(identity_channels)}')
print(f'Functional channels: {len(functional_channels)}')
print(f'Extra atlas mappings: {len(extra_mapping)}')

## 2. Model Inventory

One model is saved per *(study, target)* pair. The winning model type was selected by 5-fold CV R².

In [ ]:
inventory = load_model_inventory(MODELS_DIR)

if not inventory:
    print('No models found in', MODELS_DIR)
    print('Run  scripts/train_atlas_models.py  first.')
else:
    df_inv = pd.DataFrame([
        {
            'study':         m['study'],
            'target':        m['target_channel'],
            'model_type':    m['model_type'],
            'cv_R²':         m['cv_r2'],
            'cv_R²_std':     m.get('cv_r2_std', float('nan')),
            'test_R²':       m['test_r2'],
            'test_MAE':      m['test_mae'],
            'n_features':    len(m['input_channels']),
            'n_train':       m['n_train'],
            'n_test':        m.get('n_test', '—'),
            'atlas_version': m.get('atlas_version', '—'),
        }
        for m in inventory
    ])

    display(
        df_inv.style
        .background_gradient(subset=['cv_R²', 'test_R²'], cmap='RdYlGn', vmin=0, vmax=1)
        .background_gradient(subset=['test_MAE'], cmap='RdYlGn_r', vmin=0, vmax=0.5)
        .bar(subset=['cv_R²_std'], color='#aec6e8', vmin=0, vmax=0.02)
        .format({
            'cv_R²': '{:.4f}', 'cv_R²_std': '{:.4f}',
            'test_R²': '{:.4f}', 'test_MAE': '{:.4f}',
            'n_train': '{:,}', 'n_test': '{:,}',
        })
    )
    print(f'\nTotal models: {len(inventory)}')

In [ ]:
if 'df_inv' in vars() and len(df_inv):
    fig, ax = plt.subplots(figsize=(max(7, len(df_inv) * 1.8), 4.5))

    x = np.arange(len(df_inv))
    w = 0.35
    colors = [model_color(mt) for mt in df_inv['model_type']]

    # CV bars with ±1 std error band
    bars_cv = ax.bar(x - w/2, df_inv['cv_R²'], width=w, color=colors, alpha=0.85,
                     label='CV R²', edgecolor='k', linewidth=0.4)
    ax.errorbar(x - w/2, df_inv['cv_R²'], yerr=df_inv['cv_R²_std'],
                fmt='none', ecolor='#333', elinewidth=1.2, capsize=4)

    # Test bars
    bars_ts = ax.bar(x + w/2, df_inv['test_R²'], width=w, color=colors, alpha=0.4,
                     label='Test R²', edgecolor='k', linewidth=0.4, hatch='//')

    # Value labels
    for bar in bars_cv:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8)
    for bar in bars_ts:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8, style='italic')

    xlabels = [f"{r['study'].split('_')[0]}\n{r['target']}" for _, r in df_inv.iterrows()]
    ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_ylim(0, min(1.15, df_inv[['cv_R²', 'test_R²']].max().max() + 0.2))
    ax.set_ylabel('R²')
    ax.set_title('CV R² (±1 std) vs Test R² per trained model', fontsize=12)

    # Legend: model types
    type_handles = [mpatches.Patch(color=model_color(mt), label=mt)
                    for mt in sorted(df_inv['model_type'].unique())]
    style_handles = [
        mpatches.Patch(color='grey', alpha=0.85, label='CV R² (solid)'),
        mpatches.Patch(color='grey', alpha=0.4,  hatch='//', label='Test R² (hatched)'),
    ]
    ax.legend(handles=type_handles + style_handles, fontsize=8, ncol=3,
              loc='upper left', framealpha=0.9)

    plt.tight_layout()
    plt.show()

## 3. Model Comparison

Cross-study overview: which model type wins, how close CV vs test scores are (overfitting check), and whether any model is systematically over- or underfit.

In [ ]:
if 'df_inv' in vars() and len(df_inv):
    model_types = sorted(df_inv['model_type'].unique())
    palette     = {mt: model_color(mt) for mt in model_types}

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # ── Left: CV R² vs Test R² scatter ───────────────────────────────────────
    ax = axes[0]
    for mt in model_types:
        sub = df_inv[df_inv['model_type'] == mt]
        ax.errorbar(sub['cv_R²'], sub['test_R²'],
                    xerr=sub['cv_R²_std'],
                    fmt='o', markersize=10, color=palette[mt],
                    markeredgecolor='k', markeredgewidth=0.5,
                    ecolor=palette[mt], elinewidth=1.2, capsize=4,
                    label=mt, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(row['target'], (row['cv_R²'], row['test_R²']),
                        textcoords='offset points', xytext=(6, 3),
                        fontsize=8, color='#333')
    lo = df_inv[['cv_R²', 'test_R²']].min().min() - 0.06
    hi = df_inv[['cv_R²', 'test_R²']].max().max() + 0.06
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.2, alpha=0.35, label='no overfit (CV = test)')
    ax.fill_between([lo, hi], [lo - 0.03, hi - 0.03], [lo, hi],
                    alpha=0.05, color='red', label='overfit zone')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel('CV R² (± 1 std error bar)'); ax.set_ylabel('Test R²')
    ax.set_title('CV vs Test R²\n(distance from diagonal = overfitting)', fontsize=10)
    ax.legend(fontsize=7, framealpha=0.9)

    # ── Middle: grouped bar per target ────────────────────────────────────────
    ax = axes[1]
    xlabels = [f"{r['study'].split('_')[0]}\n{r['target']}" for _, r in df_inv.iterrows()]
    x = np.arange(len(df_inv))
    w = 0.38
    colours = [palette[mt] for mt in df_inv['model_type']]
    ax.bar(x - w/2, df_inv['cv_R²'],  width=w, color=colours, alpha=0.85)
    ax.bar(x + w/2, df_inv['test_R²'], width=w, color=colours, alpha=0.40,
           hatch='//', edgecolor='k', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_ylim(0, min(1.05, df_inv[['cv_R²', 'test_R²']].max().max() + 0.15))
    ax.set_ylabel('R²')
    ax.set_title('CV R² vs Test R² per model\n(colour = winning model type)', fontsize=10)
    type_handles  = [mpatches.Patch(color=palette[mt], label=mt) for mt in model_types]
    style_handles = [mpatches.Patch(color='grey', alpha=0.85, label='CV (solid)'),
                     mpatches.Patch(color='grey', alpha=0.40, hatch='//', label='Test (hatched)')]
    ax.legend(handles=type_handles + style_handles, fontsize=7, ncol=2, framealpha=0.9)

    # ── Right: win count donut ────────────────────────────────────────────────
    ax = axes[2]
    wins = df_inv['model_type'].value_counts().reindex(model_types, fill_value=0)
    wins = wins[wins > 0]
    wedge_colors = [palette[mt] for mt in wins.index]
    wedges, texts, autotexts = ax.pie(
        wins.values, labels=wins.index, colors=wedge_colors,
        autopct=lambda p: f'{p:.0f}%\n({int(round(p*len(df_inv)/100))})',
        startangle=90, pctdistance=0.65,
        wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
    for at in autotexts:
        at.set(fontsize=9, fontweight='bold', color='white')
    for t in texts:
        t.set(fontsize=9)
    ax.set_title('Model-type win share\n(targets won)', fontsize=10)

    plt.suptitle('Model selection comparison', fontsize=13, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── Overfit gap table ─────────────────────────────────────────────────────
    df_inv['overfit_gap'] = df_inv['cv_R²'] - df_inv['test_R²']
    print('Overfit gap (CV R² − Test R²) — values near 0 = no overfitting:')
    display(
        df_inv[['study', 'target', 'model_type', 'cv_R²', 'test_R²', 'overfit_gap']]
        .sort_values('overfit_gap', ascending=False)
        .style
        .background_gradient(subset=['overfit_gap'], cmap='RdYlGn_r', vmin=-0.01, vmax=0.08)
        .format({'cv_R²': '{:.4f}', 'test_R²': '{:.4f}', 'overfit_gap': '{:+.5f}'})
    )

## 4. Feature Map

Which identity markers (atlas variables) are used as inputs by each model, and how much each feature contributes on average.

In [ ]:
if 'inventory' in vars() and inventory:
    all_features = sorted({f for m in inventory for f in m['input_channels']})
    row_labels   = [f"{m['study'].split('_')[0]} / {m['target_channel']}" for m in inventory]

    feat_matrix = np.array([
        [1 if f in m['input_channels'] else 0 for f in all_features]
        for m in inventory
    ], dtype=float)

    fig, ax = plt.subplots(figsize=(max(8, len(all_features) * 0.9), max(3, len(inventory) * 0.75 + 1.5)))
    cmap = mcolors.ListedColormap(['#f0f0f0', '#1f77b4'])
    im = ax.imshow(feat_matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1)

    ax.set_xticks(range(len(all_features)))
    ax.set_xticklabels(all_features, rotation=40, ha='right', fontsize=10)
    ax.set_yticks(range(len(inventory)))
    ax.set_yticklabels(row_labels, fontsize=10)

    # Model type label on the right
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(range(len(inventory)))
    ax2.set_yticklabels(
        [m['model_type'] for m in inventory],
        fontsize=9)
    ax2.tick_params(axis='y', length=0)
    ax2.set_ylabel('Winning model type', fontsize=9, labelpad=8)

    # Cell annotations
    for i in range(len(inventory)):
        for j in range(len(all_features)):
            val = feat_matrix[i, j]
            ax.text(j, i, '✓' if val else '', ha='center', va='center',
                    fontsize=11, color='white' if val else '#bbb')

    ax.set_title('Feature usage map — identity markers as model inputs', fontsize=12, pad=10)
    ax.set_xlabel('Atlas identity marker (input feature)')
    plt.tight_layout()
    plt.show()

    # Feature frequency bar
    freq = feat_matrix.sum(axis=0) / len(inventory) * 100
    fig2, ax2 = plt.subplots(figsize=(max(6, len(all_features) * 0.9), 2.8))
    ax2.bar(all_features, freq, color='#1f77b4', edgecolor='white', linewidth=0.5)
    ax2.set_ylabel('% models using feature')
    ax2.set_ylim(0, 115)
    ax2.set_xticklabels(all_features, rotation=40, ha='right', fontsize=10)
    for i, v in enumerate(freq):
        ax2.text(i, v + 1, f'{v:.0f}%', ha='center', fontsize=9)
    ax2.set_title('Feature usage frequency across all trained models')
    plt.tight_layout()
    plt.show()

## 5. Held-out Atlas Evaluation

Sample unseen atlas cells and compare ONNX model predictions to ground-truth measured intensities. All sampled cells are *outside* the training set (we subsample a fresh random draw).

In [ ]:
print('Opening atlas (backed mode — fast)…')
adata_backed   = ad.read_h5ad(ATLAS_PATH, backed='r')
atlas_var_names = list(adata_backed.var_names)
n_atlas_cells  = adata_backed.n_obs
print(f'Atlas: {n_atlas_cells:,} cells × {len(atlas_var_names):,} vars')

spt_to_atlas = build_spt_to_atlas(atlas_var_names, all_channels, aliases, extra_mapping)
print(f'SPT ↔ atlas channel matches: {len(spt_to_atlas)}')

In [ ]:
EVAL_STUDY  = PRIMARY_STUDY
EVAL_TARGET = None   # None → evaluate all models for this study

study_models = [m for m in inventory if m['study'] == EVAL_STUDY]
if EVAL_TARGET:
    study_models = [m for m in study_models if m['target_channel'] == EVAL_TARGET]

if not study_models:
    print(f'No models for study={EVAL_STUDY} target={EVAL_TARGET}')
else:
    # Collect all SPT channel names needed
    needed_spt = set()
    for m in study_models:
        needed_spt.update(m['input_channels'])
        needed_spt.add(m['target_channel'])

    # Map to atlas names, keep only present
    pairs = [(spt_to_atlas[s], s) for s in needed_spt if s in spt_to_atlas]
    needed_atlas = [p[0] for p in pairs]
    present_spt  = [p[1] for p in pairs]

    SAMPLE_SIZE = 50_000
    rng = np.random.default_rng(42)
    idx = rng.choice(n_atlas_cells, size=min(SAMPLE_SIZE, n_atlas_cells), replace=False)
    idx.sort()

    print(f'Loading {len(needed_atlas)} atlas columns for {len(idx):,} sampled cells…')
    subset = adata_backed[idx, needed_atlas]
    X_eval = subset.X
    if sparse.issparse(X_eval):
        X_eval = X_eval.toarray()
    X_eval = np.asarray(X_eval, dtype=np.float32)

    col_idx = {spt: i for i, spt in enumerate(present_spt)}
    print('Done. Eval matrix shape:', X_eval.shape)

In [ ]:
if 'X_eval' in vars() and 'study_models' in vars():
    n = len(study_models)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, meta in zip(axes, study_models):
        target   = meta['target_channel']
        features = meta['input_channels']
        feat_ok  = [f for f in features if f in col_idx]

        if target not in col_idx or not feat_ok:
            ax.set_title(f'{target}\n(missing data)'); continue

        X_feat = X_eval[:, [col_idx[f] for f in feat_ok]]
        y_true = X_eval[:, col_idx[target]]

        # Sum-normalize: divide each cell by sum of its identity channels
        S_feat = X_feat.sum(axis=1)
        valid  = S_feat > 1e-8
        X_feat_norm = X_feat[valid] / S_feat[valid, np.newaxis]
        y_true_norm = y_true[valid] / S_feat[valid]
        y_pred      = run_onnx(meta['onnx_path'], X_feat_norm)

        r2  = r2_score(y_true_norm, y_pred)
        mae = mean_absolute_error(y_true_norm, y_pred)

        lo = min(y_true_norm.min(), y_pred.min())
        hi = max(y_true_norm.max(), y_pred.max())

        hb = ax.hexbin(y_true_norm, y_pred, gridsize=45, cmap='Blues',
                       mincnt=1, linewidths=0.1)
        plt.colorbar(hb, ax=ax, label='cell count')
        ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='perfect')
        ax.set_xlabel('Measured / S (sum-normalized)')
        ax.set_ylabel('ONNX Predicted (sum-normalized)')
        ax.set_title(f'{target}  [{meta["model_type"]}]\nR²={r2:.4f}   MAE={mae:.4f}'
                     f'  (n={valid.sum():,}, {(~valid).sum()} zero-sum excluded)')
        ax.legend(fontsize=8)

    fig.suptitle(f'Atlas hold-out: predicted vs. measured (sum-normalized) — {EVAL_STUDY}',
                 fontsize=12, y=1.01, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'X_eval' in vars() and 'study_models' in vars():
    n = len(study_models)
    fig, axes = plt.subplots(2, n, figsize=(5.5 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for j, meta in enumerate(study_models):
        target   = meta['target_channel']
        features = meta['input_channels']
        feat_ok  = [f for f in features if f in col_idx]

        if target not in col_idx or not feat_ok:
            axes[0, j].set_visible(False); axes[1, j].set_visible(False); continue

        X_feat = X_eval[:, [col_idx[f] for f in feat_ok]]
        y_true = X_eval[:, col_idx[target]]

        S_feat = X_feat.sum(axis=1)
        valid  = S_feat > 1e-8
        X_feat_norm = X_feat[valid] / S_feat[valid, np.newaxis]
        y_true_norm = y_true[valid] / S_feat[valid]

        y_pred    = run_onnx(meta['onnx_path'], X_feat_norm)
        residuals = y_true_norm - y_pred
        color     = model_color(meta['model_type'])

        # Row 0: residual histogram + KDE
        ax = axes[0, j]
        ax.hist(residuals, bins=80, density=True, color=color, alpha=0.5, edgecolor='none')
        xg  = np.linspace(residuals.min(), residuals.max(), 300)
        kde = stats.gaussian_kde(residuals)
        ax.plot(xg, kde(xg), color=color, lw=2)
        ax.axvline(0, color='red', lw=1.5, linestyle='--')
        ax.set_xlabel('Residual (measured/S − predicted)')
        ax.set_ylabel('Density')
        ax.set_title(f'{target} [{meta["model_type"]}]\nResidual distribution  '
                     f'μ={residuals.mean():.4f}  σ={residuals.std():.4f}')

        # Row 1: Q–Q plot
        ax = axes[1, j]
        (osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
        ax.scatter(osm, osr, s=1, alpha=0.3, color=color)
        ax.plot(osm, slope * np.array(osm) + intercept, 'r--', lw=1.5)
        ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Sample quantiles')
        ax.set_title(f'{target} — Q–Q plot  (R={r:.4f})')

    fig.suptitle(f'Residual analysis (sum-normalized) — {EVAL_STUDY}',
                 fontsize=12, y=1.01, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Extended prediction diagnostics per model ─────────────────────────────────
# 1. Residual vs. predicted  — checks heteroscedasticity
# 2. Calibration curve       — binned predicted vs. actual mean
# 3. Marginal distributions  — measured vs. predicted KDE
if 'X_eval' in vars() and 'study_models' in vars():
    N_BINS = 20

    for meta in study_models:
        target  = meta['target_channel']
        feat_ok = [f for f in meta['input_channels'] if f in col_idx]
        if target not in col_idx or not feat_ok:
            continue

        X_feat = X_eval[:, [col_idx[f] for f in feat_ok]]
        y_true = X_eval[:, col_idx[target]]

        S_feat = X_feat.sum(axis=1)
        valid  = S_feat > 1e-8
        X_feat_norm = X_feat[valid] / S_feat[valid, np.newaxis]
        y_true_norm = y_true[valid] / S_feat[valid]

        y_pred    = run_onnx(meta['onnx_path'], X_feat_norm)
        residuals = y_true_norm - y_pred
        color     = model_color(meta['model_type'])

        fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

        # ── Panel 1: Residual vs. predicted ───────────────────────────────────
        ax = axes[0]
        ax.scatter(y_pred, residuals, alpha=0.015, s=1, color=color, rasterized=True)
        ax.axhline(0, color='red', lw=1.5, linestyle='--', label='zero')
        coeffs = np.polyfit(y_pred, residuals, 2)
        xg     = np.linspace(y_pred.min(), y_pred.max(), 200)
        ax.plot(xg, np.polyval(coeffs, xg), 'k-', lw=2, alpha=0.8, label='trend (poly-2)')
        q_edges = np.percentile(y_pred, np.linspace(0, 100, N_BINS + 1))
        bin_c, bin_std = [], []
        for lo_q, hi_q in zip(q_edges[:-1], q_edges[1:]):
            m_ = (y_pred >= lo_q) & (y_pred < hi_q)
            if m_.sum() >= 5:
                bin_c.append(y_pred[m_].mean())
                bin_std.append(residuals[m_].std())
        bin_c, bin_std = np.array(bin_c), np.array(bin_std)
        ax.fill_between(bin_c, -bin_std, bin_std, alpha=0.12, color=color, label='±1 std spread')
        ax.set_xlabel('Predicted (sum-normalized)')
        ax.set_ylabel('Residual (measured/S − predicted)')
        ax.set_title(f'{target} [{meta["model_type"]}]\nResidual vs. Predicted\n'
                     f'(flat trend + uniform spread = good)')
        ax.legend(fontsize=8)

        # ── Panel 2: Calibration curve ────────────────────────────────────────
        ax = axes[1]
        bin_means_pred, bin_means_true, bin_std_true = [], [], []
        for lo_q, hi_q in zip(q_edges[:-1], q_edges[1:]):
            m_ = (y_pred >= lo_q) & (y_pred < hi_q)
            if m_.sum() < 5:
                continue
            bin_means_pred.append(y_pred[m_].mean())
            bin_means_true.append(y_true_norm[m_].mean())
            bin_std_true.append(y_true_norm[m_].std())
        bp = np.array(bin_means_pred)
        bt = np.array(bin_means_true)
        bs = np.array(bin_std_true)
        ax.fill_between(bp, bt - bs, bt + bs, alpha=0.18, color=color, label='±1 std (actual)')
        ax.plot(bp, bt, 'o-', color=color, lw=2, markersize=5, label='Mean actual per bin')
        lo_d = min(bp.min(), bt.min()) - 0.02
        hi_d = max(bp.max(), bt.max()) + 0.02
        ax.plot([lo_d, hi_d], [lo_d, hi_d], 'r--', lw=1.5, label='Perfect calibration')
        ax.set_xlabel(f'Mean predicted ({N_BINS} equal-freq. bins)')
        ax.set_ylabel('Mean actual (sum-normalized)')
        ax.set_title(f'{target} [{meta["model_type"]}]\nCalibration curve\n'
                     f'(points on diagonal = well-calibrated)')
        ax.legend(fontsize=8)

        # ── Panel 3: Marginal distributions ──────────────────────────────────
        ax = axes[2]
        xg = np.linspace(min(y_true_norm.min(), y_pred.min()),
                         max(y_true_norm.max(), y_pred.max()), 400)
        for vals, lbl, col in [
            (y_true_norm, 'Measured/S', '#1f77b4'),
            (y_pred,      'Predicted',  color),
        ]:
            kde_fn = stats.gaussian_kde(vals, bw_method='scott')
            ax.plot(xg, kde_fn(xg), color=col, lw=2.5, label=lbl)
            ax.fill_between(xg, kde_fn(xg), alpha=0.12, color=col)
        ax.set_xlabel('Sum-normalized intensity')
        ax.set_ylabel('Density')
        ax.set_title(f'{target} [{meta["model_type"]}]\nMarginal distributions\n'
                     f'(overlap ↔ model captures intensity range)')
        ax.legend(fontsize=9)

        fig.suptitle(f'{target}  [{meta["model_type"]}] — prediction diagnostics (sum-normalized)',
                     fontsize=12, y=1.01, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Atlas hold-out z-score calibration ───────────────────────────────────────
# Well-calibrated uncertainty → z-scores ~ N(0,1).  |z| > 2 ≈ 4.6% expected.
if 'X_eval' in vars() and 'study_models' in vars():
    for meta in study_models:
        target  = meta['target_channel']
        feat_ok = [f for f in meta['input_channels'] if f in col_idx]
        if target not in col_idx or not feat_ok:
            continue

        X_feat = X_eval[:, [col_idx[f] for f in feat_ok]]
        y_true = X_eval[:, col_idx[target]]

        S_feat = X_feat.sum(axis=1)
        valid  = S_feat > 1e-8
        X_feat_norm = X_feat[valid] / S_feat[valid, np.newaxis]
        y_true_norm = y_true[valid] / S_feat[valid]

        base   = meta['meta_path'].name.split('.meta.json')[0]
        pkl_path = meta['meta_path'].parent / (base + '.pkl')
        if not pkl_path.exists():
            print(f'[{target}] No .pkl — skipping z-score panel.')
            continue

        model        = load_pkl(pkl_path)
        y_pred, y_std = predict_with_std(model, meta['model_type'], X_feat_norm)

        if y_std is None:
            g_std = meta.get('global_std')
            if g_std:
                y_std = np.full_like(y_pred, g_std)
            else:
                print(f'[{target}] No uncertainty estimate — skipping.')
                continue

        z = (y_true_norm - y_pred) / np.maximum(y_std, 1e-12)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        color = model_color(meta['model_type'])

        # Panel 1: z-score histogram vs N(0,1)
        ax = axes[0]
        ax.hist(z, bins=80, density=True, color=color, alpha=0.55, edgecolor='none',
                label='z-scores')
        xg = np.linspace(-5, 5, 300)
        ax.plot(xg, stats.norm.pdf(xg), 'r-', lw=2, label='N(0,1)')
        ax.axvline(-2, color='grey', lw=1, linestyle='--')
        ax.axvline( 2, color='grey', lw=1, linestyle='--')
        frac_out = (np.abs(z) > 2).mean()
        ax.set_xlim(-6, 6)
        ax.set_xlabel('z-score  (measured/S − predicted) / σ')
        ax.set_ylabel('Density')
        ax.set_title(f'{target} [{meta["model_type"]}]\n'
                     f'z-score distribution  |z|>2: {frac_out*100:.1f}%'
                     f'  (N(0,1) expected ~4.6%)')
        ax.legend(fontsize=9)

        # Panel 2: z-score vs predicted
        ax = axes[1]
        ax.scatter(y_pred, z, alpha=0.02, s=1, color=color, rasterized=True)
        ax.axhline( 0, color='red',  lw=1,   linestyle='--')
        ax.axhline(-2, color='grey', lw=0.8, linestyle=':')
        ax.axhline( 2, color='grey', lw=0.8, linestyle=':')
        ax.set_xlabel('Predicted (sum-normalized)')
        ax.set_ylabel('z-score')
        ax.set_title(f'{target} — z-score vs predicted\n'
                     f'(uniform band = well-calibrated uncertainty)')

        fig.suptitle(f'{target} [{meta["model_type"]}] — uncertainty calibration (atlas hold-out)',
                     fontsize=12, y=1.01, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print(f'[{target}] std_method={meta.get("std_method","?"):25s}  '
              f'|z|>2: {frac_out*100:.1f}%  median_σ={np.median(y_std):.5f}')

## 6. Per-cell-type Residual Analysis

Check whether prediction residuals are systematically biased for specific cell populations. Large biases may indicate that a cell type is under-represented in the training data or has a biochemically distinct marker relationship.

In [ ]:
obs_cols = list(adata_backed.obs.columns)
print('Atlas obs columns:', obs_cols)

CELL_TYPE_COL = None
for candidate in ('cell_type', 'celltype', 'cell_type_l1', 'cluster', 'leiden', 'louvain'):
    if candidate in obs_cols:
        CELL_TYPE_COL = candidate
        break

if CELL_TYPE_COL:
    print(f'Using cell-type column: {CELL_TYPE_COL!r}')
    cell_types = adata_backed.obs[CELL_TYPE_COL].iloc[idx].values
else:
    print('No cell-type column found — per-cell-type analysis will be skipped.')
    cell_types = None

In [ ]:
if cell_types is not None and 'study_models' in vars() and study_models:
    TOP_N = 14

    for meta in study_models:
        target   = meta['target_channel']
        features = meta['input_channels']
        feat_ok  = [f for f in features if f in col_idx]
        if target not in col_idx or not feat_ok:
            continue

        X_feat    = X_eval[:, [col_idx[f] for f in feat_ok]]
        y_true    = X_eval[:, col_idx[target]]
        y_pred    = run_onnx(meta['onnx_path'], X_feat)
        residuals = y_true - y_pred

        df_res = pd.DataFrame({'cell_type': cell_types, 'residual': residuals})
        top_ct = df_res['cell_type'].value_counts().head(TOP_N).index
        df_res = df_res[df_res['cell_type'].isin(top_ct)]

        order = df_res.groupby('cell_type')['residual'].median().sort_values().index

        fig, ax = plt.subplots(figsize=(11, 4.5))
        sns.violinplot(data=df_res, x='cell_type', y='residual', order=order,
                       ax=ax, inner='box', palette='Set2', cut=0)
        ax.axhline(0, color='red', linestyle='--', lw=1.2, alpha=0.7)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
        ax.set_title(f'Residuals by cell type — {target}  [{meta["model_type"]}]\n'
                     f'(top {TOP_N} cell types by count)', fontsize=11)
        ax.set_ylabel('Measured − Predicted')
        plt.tight_layout()
        plt.show()

        # Median residual heatmap across targets
else:
    if cell_types is None:
        print('Skipped: no cell-type annotation found in atlas obs.')

In [ ]:
if cell_types is not None and 'study_models' in vars() and len(study_models) > 1:
    TOP_N = 15
    # Build a matrix: cell_types × targets
    ct_series = pd.Series(cell_types)
    top_ct    = ct_series.value_counts().head(TOP_N).index.tolist()

    heat_data = {}
    for meta in study_models:
        target   = meta['target_channel']
        feat_ok  = [f for f in meta['input_channels'] if f in col_idx]
        if target not in col_idx or not feat_ok:
            continue
        X_feat = X_eval[:, [col_idx[f] for f in feat_ok]]
        y_true = X_eval[:, col_idx[target]]
        y_pred = run_onnx(meta['onnx_path'], X_feat)
        res    = y_true - y_pred
        heat_data[target] = {ct: res[ct_series == ct].mean() for ct in top_ct}

    if heat_data:
        df_heat = pd.DataFrame(heat_data, index=top_ct)
        fig, ax = plt.subplots(figsize=(max(5, len(heat_data) * 1.8), max(4, TOP_N * 0.45)))
        sns.heatmap(df_heat, ax=ax, cmap='RdBu_r', center=0,
                    annot=True, fmt='.3f', linewidths=0.4,
                    cbar_kws={'label': 'Mean residual (measured − predicted)'})
        ax.set_xlabel('Target channel'); ax.set_ylabel('Cell type')
        ax.set_title(f'Mean residual per cell-type × target — {EVAL_STUDY}', fontsize=11)
        plt.tight_layout()
        plt.show()

## 7. SPT Study Inference

Run the atlas-reference models on actual SPT study cell data. Each cell receives a *predicted* functional marker value; cells where measured > predicted are atlas-relative **positive**.

In [ ]:
study_dir = DATASETS_DIR / PRIMARY_STUDY
expr_candidates = [
    study_dir / 'generated_artifacts' / 'cells.h5ad',
    study_dir / 'cells.h5ad',
    study_dir / 'expression.h5ad',
]
spt_adata_path = next((p for p in expr_candidates if p.exists()), None)

if spt_adata_path is None:
    print(f'No cell expression h5ad found for {PRIMARY_STUDY}.')
    print('Files present:', sorted(study_dir.rglob('*.h5ad')))
    spt_adata = None
else:
    spt_adata = ad.read_h5ad(spt_adata_path)
    print(f'SPT study: {spt_adata.n_obs:,} cells × {spt_adata.n_vars} channels')
    print('Channels:', list(spt_adata.var_names)[:20])

In [ ]:
results = {}   # target_channel → dict

if spt_adata is not None:
    spt_var_set = set(spt_adata.var_names)

    for meta in study_models:
        target   = meta['target_channel']
        features = meta['input_channels']
        feat_ok  = [f for f in features if f in spt_var_set]

        missing = set(features) - spt_var_set
        if missing:
            print(f'[{target}] Missing features in SPT data: {missing}')
        if target not in spt_var_set:
            print(f'[{target}] Target not in SPT data — skipping.'); continue
        if not feat_ok:
            print(f'[{target}] No features in SPT data — skipping.');  continue

        def _to_dense(X):
            return X.toarray() if sparse.issparse(X) else np.asarray(X)

        X_spt      = _to_dense(spt_adata[:, feat_ok].X).astype(np.float32)
        y_measured = _to_dense(spt_adata[:, target].X).flatten().astype(np.float32)

        # Sum-normalize by identity-channel sum
        s          = X_spt.sum(axis=1)
        valid_mask = s > 1e-8
        n_excluded = int((~valid_mask).sum())
        if n_excluded:
            print(f'[{target}] Excluded {n_excluded:,} cells with zero identity-channel sum.')
        X_spt_norm = X_spt[valid_mask] / s[valid_mask, np.newaxis]
        y_norm     = y_measured[valid_mask] / s[valid_mask]

        y_predicted = run_onnx(meta['onnx_path'], X_spt_norm)
        pos_mask    = y_norm > y_predicted

        # Per-sample uncertainty
        y_std    = None
        z_scores = None
        base     = meta['meta_path'].name.split('.meta.json')[0]
        pkl_path = meta['meta_path'].parent / (base + '.pkl')
        if pkl_path.exists():
            model    = load_pkl(pkl_path)
            _, y_std = predict_with_std(model, meta['model_type'], X_spt_norm)
            if y_std is None:
                g_std = meta.get('global_std')
                if g_std:
                    y_std = np.full(y_norm.shape, float(g_std), dtype=np.float32)
        if y_std is not None:
            z_scores = (y_norm - y_predicted) / np.maximum(y_std, 1e-12)

        results[target] = {
            'measured':                y_measured,
            'y_norm':                  y_norm,
            'predicted_raw':           y_predicted,
            'valid_mask':              valid_mask,
            'atlas_relative_positive': pos_mask,
            'model_type':              meta['model_type'],
            'y_std':                   y_std,
            'z_scores':                z_scores,
        }
        r2_val = r2_score(y_norm, y_predicted) if len(y_norm) > 1 else float('nan')
        z_str  = (f'  |z|>2: {(np.abs(z_scores) > 2).mean()*100:.1f}%'
                  if z_scores is not None else '')
        print(f'{target:10s} [{meta["model_type"]:15s}]  '
              f'atlas-relative positive: {pos_mask.mean()*100:.1f}%  '
              f'(R²={r2_val:.4f}){z_str}')

    if not results:
        print('\nNo SPT inference results. Check that study data exists.')
    else:
        print('\nInference complete.')

## 8. Atlas-relative Positive Calls: Distribution Comparison

Compare the measured intensity distributions of atlas-relative **positive** (marker higher than healthy reference) vs. **negative** cells. A clear separation validates the classifier.

In [ ]:
if results:
    n_t = len(results)
    fig, axes = plt.subplots(1, n_t, figsize=(5.5 * n_t, 4.5))
    if n_t == 1:
        axes = [axes]

    pos_rates = {}
    for ax, (target, res) in zip(axes, results.items()):
        pos   = res['atlas_relative_positive']
        neg   = ~pos
        y_nm  = res['y_norm']         # sum-normalised measured values
        y_pr  = res['predicted_raw']  # sum-normalised predicted values
        color = model_color(res['model_type'])

        for mask, lbl, col, lw in [
            (neg, 'Atlas-rel. negative', '#4878CF', 2.5),
            (pos, 'Atlas-rel. positive', '#D65F5F', 2.5),
        ]:
            vals = y_nm[mask]
            if len(vals) > 10:
                kde = stats.gaussian_kde(vals, bw_method='scott')
                xg  = np.linspace(y_nm.min(), y_nm.max(), 400)
                ax.plot(xg, kde(xg), color=col, lw=lw, label=f'{lbl} (n={mask.sum():,})')
                ax.fill_between(xg, kde(xg), alpha=0.18, color=col)

        ax.axvline(y_pr.mean(), color='k', lw=1, linestyle=':',
                   label=f'mean predicted={y_pr.mean():.3f}')
        ax.set_xlabel(f'{target} measured / S (sum-normalized)')
        ax.set_ylabel('Density')
        pct = pos.mean() * 100
        pos_rates[target] = pct
        ax.set_title(f'{target}  [{res["model_type"]}]\n{pct:.1f}% atlas-relative positive',
                     fontsize=10)
        ax.legend(fontsize=7, framealpha=0.9)

    fig.suptitle(f'{PRIMARY_STUDY} — atlas-relative positive/negative distributions (sum-normalized)',
                 fontsize=12, y=1.01, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Summary positive rates bar
    fig2, ax2 = plt.subplots(figsize=(max(5, n_t * 1.8), 3.2))
    bar_colors = [model_color(results[t]['model_type']) for t in pos_rates]
    bars = ax2.bar(list(pos_rates.keys()), list(pos_rates.values()),
                   color=bar_colors, edgecolor='k', linewidth=0.5)
    for bar, v in zip(bars, pos_rates.values()):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 0.5, f'{v:.1f}%',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax2.axhline(50, color='grey', lw=1, linestyle='--', alpha=0.5, label='50% line')
    ax2.set_ylim(0, max(pos_rates.values()) + 12)
    ax2.set_ylabel('% atlas-relative positive')
    ax2.set_title('Atlas-relative positive rate per target')
    type_handles = [mpatches.Patch(color=model_color(results[t]['model_type']),
                    label=results[t]['model_type']) for t in results]
    ax2.legend(handles=type_handles, fontsize=8, title='Model type')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── SPT z-score distribution ──────────────────────────────────────────────────
if results:
    n_t = len(results)
    has_z = any(res.get('z_scores') is not None for res in results.values())
    if not has_z:
        print('No uncertainty estimates available — skipping z-score plot.')
    else:
        fig, axes = plt.subplots(1, n_t, figsize=(6 * n_t, 4.5))
        if n_t == 1:
            axes = [axes]

        for ax, (target, res) in zip(axes, results.items()):
            z = res.get('z_scores')
            if z is None:
                ax.set_title(f'{target}\n(no uncertainty estimate)'); continue
            color = model_color(res['model_type'])
            ax.hist(z, bins=80, density=True, color=color, alpha=0.55,
                    edgecolor='none', label='z-scores')
            xg = np.linspace(-5, 5, 300)
            ax.plot(xg, stats.norm.pdf(xg), 'r-', lw=2, label='N(0,1)')
            ax.axvline(-2, color='grey', lw=1, linestyle='--')
            ax.axvline( 2, color='grey', lw=1, linestyle='--')
            frac_out = (np.abs(z) > 2).mean()
            ax.set_xlim(-6, 6)
            ax.set_xlabel('z-score  (y/s − ŷ) / σ')
            ax.set_ylabel('Density')
            ax.set_title(f'{target} [{res["model_type"]}]\n'
                         f'SPT z-scores  |z|>2: {frac_out*100:.1f}%')
            ax.legend(fontsize=9)

        fig.suptitle(f'{PRIMARY_STUDY} — SPT z-score distributions',
                     fontsize=12, y=1.01, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Summary table
        print(f'{"Target":20s}  {"Model":18s}  {"z>+2 (unexpect. high)":>22s}  {"z<-2 (unexpect. low)":>21s}')
        for target, res in results.items():
            z = res.get('z_scores')
            if z is None:
                continue
            frac_hi = (z >  2).mean() * 100
            frac_lo = (z < -2).mean() * 100
            print(f'{target:20s}  {res["model_type"]:18s}  {frac_hi:22.1f}%  {frac_lo:21.1f}%')

## 9. Biological Sanity Check

Verify that atlas-relative positive calls are enriched in **biologically expected** cell types:

| Target | Expected positive in |
|--------|----------------------|
| ITGAX (CD11c) | Dendritic cells, macrophages |
| FCGR3A (CD16) | NK cells, non-classical monocytes |
| HLA-DR | Antigen-presenting cells (macrophages, B cells, DCs) |

In [ ]:
if spt_adata is not None and results:
    cell_type_col = None
    for candidate in ('cell_type', 'celltype', 'cluster', 'phenotype', 'leiden', 'Cell Type'):
        if candidate in spt_adata.obs.columns:
            cell_type_col = candidate
            break

    if cell_type_col is None:
        print('No cell-type column in SPT obs:', list(spt_adata.obs.columns))
        print('Skipping biological sanity check.')
    else:
        spt_cts_all = spt_adata.obs[cell_type_col].values
        n_t         = len(results)

        fig, axes = plt.subplots(1, n_t, figsize=(6 * n_t, 5))
        if n_t == 1:
            axes = [axes]

        for ax, (target, res) in zip(axes, results.items()):
            # Align cell-type labels with the valid (non-zero-sum) subset
            spt_cts = spt_cts_all[res['valid_mask']]
            pos     = res['atlas_relative_positive']

            df_ct = pd.DataFrame({'cell_type': spt_cts, 'positive': pos.astype(int)})
            pct_by = (df_ct.groupby('cell_type')['positive']
                      .agg(['mean', 'count'])
                      .rename(columns={'mean': 'pct', 'count': 'n'})
                      .query('n >= 20')
                      .sort_values('pct', ascending=True)
                      .tail(20))
            pct_by['pct'] *= 100

            color = model_color(res['model_type'])
            y_pos = range(len(pct_by))
            ax.barh(y_pos, pct_by['pct'], color=color, alpha=0.75, edgecolor='k', lw=0.3)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(pct_by.index, fontsize=8)
            for i, (_, row) in enumerate(pct_by.iterrows()):
                ax.text(row['pct'] + 0.3, i, f'{row["pct"]:.1f}%  (n={int(row["n"]):,})',
                        va='center', fontsize=7, color='#333')
            ax.set_xlabel('% atlas-relative positive')
            ax.set_xlim(0, pct_by['pct'].max() + 12)
            ax.axvline(res['atlas_relative_positive'].mean() * 100,
                       color='red', lw=1.2, linestyle='--', alpha=0.6,
                       label=f'overall avg {res["atlas_relative_positive"].mean()*100:.1f}%')
            ax.legend(fontsize=7)
            ax.set_title(f'{target} [{res["model_type"]}]\n% atlas-rel. positive by cell type',
                         fontsize=10)

        plt.suptitle(f'{PRIMARY_STUDY} — biological sanity check',
                     fontsize=12, y=1.01, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 10. Summary

In [ ]:
rows = []
for meta in inventory:
    onnx_file = MODELS_DIR / meta['study'] / (meta['target_channel'].replace('/', '_') + '.onnx')
    rows.append({
        'study':         meta['study'],
        'target':        meta['target_channel'],
        'model_type':    meta['model_type'],
        'n_features':    len(meta['input_channels']),
        'CV R²':         meta['cv_r2'],
        'CV R² std':     meta.get('cv_r2_std', float('nan')),
        'Test R²':       meta['test_r2'],
        'overfit_gap':   round(meta['cv_r2'] - meta['test_r2'], 6),
        'Test MAE':      meta['test_mae'],
        'n_train':       meta['n_train'],
        'n_test':        meta.get('n_test', '—'),
        'ONNX KB':       onnx_file.stat().st_size // 1024 if onnx_file.exists() else 'N/A',
        'atlas_version': meta.get('atlas_version', '—'),
    })

if rows:
    df_sum = pd.DataFrame(rows)
    display(
        df_sum.style
        .background_gradient(subset=['CV R²', 'Test R²'], cmap='RdYlGn', vmin=0, vmax=1)
        .background_gradient(subset=['overfit_gap'],       cmap='RdYlGn_r', vmin=-0.005, vmax=0.05)
        .background_gradient(subset=['Test MAE'],          cmap='RdYlGn_r', vmin=0, vmax=0.5)
        .format({
            'CV R²': '{:.4f}', 'CV R² std': '{:.5f}',
            'Test R²': '{:.4f}', 'overfit_gap': '{:+.5f}', 'Test MAE': '{:.4f}',
            'n_train': '{:,}',
        })
    )
    print(f'\nTotal trained models: {len(rows)}')
    print('Atlas version:', df_sum['atlas_version'].unique())
else:
    print('No models trained yet.')